# 1. CSIRO Image2Biomass EDA

Deep-dive exploratory analysis for the CSIRO biomass competition. The goal is to understand the target structure, metadata signal, image signal, train-test drift, and validation risks before moving into heavier modeling.

In [ ]:
from pathlib import Path
import math
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageStat
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

PALETTE_NAME = 'viridis'
TARGET_ORDER = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
TARGET_WEIGHTS = {
    'Dry_Green_g': 0.1,
    'Dry_Dead_g': 0.1,
    'Dry_Clover_g': 0.1,
    'GDM_g': 0.2,
    'Dry_Total_g': 0.5,
}
TARGET_PALETTE = dict(zip(TARGET_ORDER, sns.color_palette(PALETTE_NAME, n_colors=len(TARGET_ORDER))))
sns.set_theme(style='whitegrid', palette=PALETTE_NAME, rc={
    'figure.figsize': (11, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

DATA_DIR = Path('/kaggle/input/competitions/csiro-biomass')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Expected Kaggle input at {DATA_DIR}')

print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

# 2. Load Data And Inspect Shape

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'train rows: {len(train):,} | images: {train["image_path"].nunique():,}')
print(f'test rows:  {len(test):,} | images: {test["image_path"].nunique():,}')
print(f'submission rows: {len(submission):,}')

display(train.head())
display(test.head())
display(train.dtypes.to_frame('dtype'))

# 3. Data Quality Checks

In [ ]:
def missing_report(df, name):
    report = (
        df.isna().sum().rename('missing')
        .to_frame()
        .assign(missing_pct=lambda x: x['missing'] / len(df))
        .query('missing > 0')
        .sort_values('missing_pct', ascending=False)
    )
    print(f'{name}: {len(report)} columns with missing values')
    display(report)

missing_report(train, 'train')
missing_report(test, 'test')

quality = pd.DataFrame({
    'check': [
        'train sample_id unique',
        'test sample_id unique',
        'train rows per image min',
        'train rows per image max',
        'test rows per image min',
        'test rows per image max',
    ],
    'value': [
        train['sample_id'].is_unique,
        test['sample_id'].is_unique,
        train.groupby('image_path').size().min(),
        train.groupby('image_path').size().max(),
        test.groupby('image_path').size().min(),
        test.groupby('image_path').size().max(),
    ],
})
display(quality)

expected_targets = set(TARGET_ORDER)
print('Unexpected train targets:', sorted(set(train['target_name']) - expected_targets))
print('Unexpected test targets:', sorted(set(test['target_name']) - expected_targets))

# 4. Metric And Target Structure

The public score is a single weighted R2 across all long-format rows. This makes `Dry_Total_g` especially important because it receives half of the total row weight.

In [ ]:
def weighted_r2_score(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mean_y = np.average(y_true, weights=weights)
    ss_res = np.sum(weights * (y_true - y_pred) ** 2)
    ss_tot = np.sum(weights * (y_true - mean_y) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0

rows_by_target = train['target_name'].value_counts().reindex(TARGET_ORDER)
metric_view = pd.DataFrame({
    'rows': rows_by_target,
    'row_weight': pd.Series(TARGET_WEIGHTS),
})
metric_view['effective_weight_mass'] = metric_view['rows'] * metric_view['row_weight']
metric_view['effective_weight_share'] = metric_view['effective_weight_mass'] / metric_view['effective_weight_mass'].sum()
display(metric_view)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=metric_view.reset_index(names='target_name'), x='target_name', y='effective_weight_share', hue='target_name', palette=TARGET_PALETTE, legend=False, ax=ax)
ax.set_title('4.1 Effective Metric Weight Share By Target')
ax.set_xlabel('')
ax.set_ylabel('share')
ax.tick_params(axis='x', rotation=25)
plt.show()

# 5. Target Distributions

In [ ]:
target_summary = (
    train.groupby('target_name')['target']
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .reindex(TARGET_ORDER)
)
display(target_summary)

g = sns.displot(
    data=train,
    x='target',
    col='target_name',
    col_wrap=3,
    hue='target_name',
    palette=TARGET_PALETTE,
    bins=45,
    facet_kws={'sharex': False, 'sharey': False},
    height=3.2,
)
g.set_titles('{col_name}')
g.fig.suptitle('5.1 Target Distributions', y=1.03)
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxenplot(data=train, x='target_name', y='target', hue='target_name', palette=TARGET_PALETTE, legend=False, ax=ax)
ax.set_title('5.2 Target Spread And Outliers')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=25)
plt.show()

# 6. Image-Level Target Composition

The long table has one row per image-target pair. Pivoting back to image level helps reveal target accounting relationships and whether `Dry_Total_g` behaves like the sum of components.

In [ ]:
image_targets = train.pivot_table(index='image_path', columns='target_name', values='target', aggfunc='first').reset_index()
meta_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
image_meta = train[meta_cols].drop_duplicates('image_path')
image_df = image_meta.merge(image_targets, on='image_path', how='left')

component_cols = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
image_df['component_sum'] = image_df[component_cols].sum(axis=1)
image_df['total_minus_components'] = image_df['Dry_Total_g'] - image_df['component_sum']
image_df['green_share_of_total'] = image_df['Dry_Green_g'] / image_df['Dry_Total_g'].replace(0, np.nan)
image_df['dead_share_of_total'] = image_df['Dry_Dead_g'] / image_df['Dry_Total_g'].replace(0, np.nan)
image_df['clover_share_of_total'] = image_df['Dry_Clover_g'] / image_df['Dry_Total_g'].replace(0, np.nan)
image_df['gdm_minus_green_clover'] = image_df['GDM_g'] - image_df[['Dry_Green_g', 'Dry_Clover_g']].sum(axis=1)

display(image_df[[
    'Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g',
    'component_sum', 'total_minus_components', 'gdm_minus_green_clover'
]].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=image_df, x='component_sum', y='Dry_Total_g', hue='Dry_Total_g', palette=PALETTE_NAME, ax=axes[0], legend=False)
limit = np.nanmax([image_df['component_sum'].max(), image_df['Dry_Total_g'].max()])
axes[0].plot([0, limit], [0, limit], color='black', linewidth=1)
axes[0].set_title('6.1 Dry Total vs Component Sum')

sns.histplot(data=image_df, x='total_minus_components', bins=50, color=sns.color_palette(PALETTE_NAME, 5)[3], ax=axes[1])
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('6.2 Dry Total Minus Component Sum')
plt.tight_layout()
plt.show()

# 7. Correlation And Redundancy

In [ ]:
corr_cols = TARGET_ORDER + ['Pre_GSHH_NDVI', 'Height_Ave_cm', 'component_sum', 'total_minus_components']
corr_cols = [c for c in corr_cols if c in image_df.columns]
corr = image_df[corr_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap=PALETTE_NAME, center=0, linewidths=0.5, ax=ax)
ax.set_title('7.1 Spearman Correlation: Targets And Numeric Metadata')
plt.show()

display(corr.round(3))

# 8. Geography, Species, And Time Coverage

In [ ]:
image_df['Sampling_Date'] = pd.to_datetime(image_df['Sampling_Date'], errors='coerce')
image_df['month'] = image_df['Sampling_Date'].dt.month
image_df['dayofyear'] = image_df['Sampling_Date'].dt.dayofyear
image_df['year_month'] = image_df['Sampling_Date'].dt.to_period('M').astype(str)
image_df['primary_species'] = image_df['Species'].fillna('Unknown').astype(str).str.split('_').str[0]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.countplot(data=image_df, x='State', hue='State', palette=PALETTE_NAME, legend=False, ax=axes[0])
axes[0].set_title('8.1 Image Count By State')
axes[0].tick_params(axis='x', rotation=25)

month_counts = image_df['month'].value_counts().sort_index().rename_axis('month').reset_index(name='images')
sns.barplot(data=month_counts, x='month', y='images', hue='month', palette=PALETTE_NAME, legend=False, ax=axes[1])
axes[1].set_title('8.2 Image Count By Month')

species_counts = image_df['primary_species'].value_counts().head(12).rename_axis('primary_species').reset_index(name='images')
sns.barplot(data=species_counts, y='primary_species', x='images', hue='primary_species', palette=PALETTE_NAME, legend=False, ax=axes[2])
axes[2].set_title('8.3 Top Primary Species')
axes[2].set_ylabel('')
plt.tight_layout()
plt.show()

display(pd.crosstab(image_df['State'], image_df['month'], margins=True))

# 9. Biomass Shifts By State, Season, And Species

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.boxenplot(data=image_df, x='State', y='Dry_Total_g', hue='State', palette=PALETTE_NAME, legend=False, ax=axes[0])
axes[0].set_title('9.1 Dry Total By State')
axes[0].tick_params(axis='x', rotation=25)

sns.boxenplot(data=image_df, x='month', y='Dry_Total_g', hue='month', palette=PALETTE_NAME, legend=False, ax=axes[1])
axes[1].set_title('9.2 Dry Total By Month')

species_order = image_df.groupby('primary_species')['Dry_Total_g'].median().sort_values(ascending=False).head(12).index
sns.boxenplot(data=image_df[image_df['primary_species'].isin(species_order)], y='primary_species', x='Dry_Total_g', order=species_order, hue='primary_species', palette=PALETTE_NAME, legend=False, ax=axes[2])
axes[2].set_title('9.3 Dry Total By Primary Species')
axes[2].set_ylabel('')
plt.tight_layout()
plt.show()

state_target = image_df.groupby('State')[TARGET_ORDER].agg(['count', 'mean', 'median']).round(2)
display(state_target)

# 10. NDVI And Height Signal

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.scatterplot(data=image_df, x='Pre_GSHH_NDVI', y='Dry_Total_g', hue='State', palette=PALETTE_NAME, ax=axes[0, 0])
axes[0, 0].set_title('10.1 NDVI vs Dry Total')

sns.scatterplot(data=image_df, x='Height_Ave_cm', y='Dry_Total_g', hue='State', palette=PALETTE_NAME, ax=axes[0, 1])
axes[0, 1].set_title('10.2 Height vs Dry Total')

sns.scatterplot(data=image_df, x='Pre_GSHH_NDVI', y='GDM_g', hue='month', palette=PALETTE_NAME, ax=axes[1, 0])
axes[1, 0].set_title('10.3 NDVI vs GDM')

sns.scatterplot(data=image_df, x='Height_Ave_cm', y='Dry_Dead_g', hue='month', palette=PALETTE_NAME, ax=axes[1, 1])
axes[1, 1].set_title('10.4 Height vs Dead Matter')
plt.tight_layout()
plt.show()

numeric_signal = image_df[['Pre_GSHH_NDVI', 'Height_Ave_cm'] + TARGET_ORDER].corr(method='spearman').loc[['Pre_GSHH_NDVI', 'Height_Ave_cm'], TARGET_ORDER]
display(numeric_signal.round(3))

# 11. Train-Test Metadata Drift

The hidden labels are unavailable, but metadata and row structure can still reveal whether validation should respect image groups, dates, states, or species.

In [ ]:
test_img = test.drop_duplicates('image_path').copy()
test_img['primary_species'] = test_img['Species'].fillna('Unknown').astype(str).str.split('_').str[0] if 'Species' in test_img else 'Unknown'
if 'Sampling_Date' in test_img:
    test_img['Sampling_Date'] = pd.to_datetime(test_img['Sampling_Date'], errors='coerce')
    test_img['month'] = test_img['Sampling_Date'].dt.month

shared_cols = [c for c in ['State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'month'] if c in image_df.columns and c in test_img.columns]
print('Comparable train-test columns:', shared_cols)

for col in shared_cols:
    if pd.api.types.is_numeric_dtype(image_df[col]) or pd.api.types.is_numeric_dtype(test_img[col]):
        fig, ax = plt.subplots(figsize=(9, 4))
        sns.kdeplot(image_df[col].dropna(), label='train', fill=True, color=sns.color_palette(PALETTE_NAME, 5)[1], ax=ax)
        sns.kdeplot(test_img[col].dropna(), label='test', fill=True, color=sns.color_palette(PALETTE_NAME, 5)[4], ax=ax)
        ax.set_title(f'11.1 Train-Test Drift: {col}')
        ax.legend()
        plt.show()
    else:
        comp = pd.concat([
            image_df[col].value_counts(normalize=True).rename('train'),
            test_img[col].value_counts(normalize=True).rename('test')
        ], axis=1).fillna(0)
        comp['abs_gap'] = (comp['train'] - comp['test']).abs()
        display(comp.sort_values('abs_gap', ascending=False).head(15))

# 12. Image Feature Extraction

These simple image descriptors are not a final vision model, but they are excellent EDA probes: color balance, visible greenness, brightness, contrast, and file dimensions often explain baseline behavior.

In [ ]:
def extract_image_features(df, cache_name):
    cache_path = OUTPUT_DIR / cache_name
    if cache_path.exists():
        return pd.read_csv(cache_path)

    rows = []
    unique_paths = df['image_path'].drop_duplicates().tolist()
    for rel_path in tqdm(unique_paths, desc=cache_name):
        path = DATA_DIR / rel_path
        row = {'image_path': rel_path}
        try:
            img = Image.open(path).convert('RGB')
            row['width'], row['height'] = img.size
            small = img.resize((256, 256))
            arr = np.asarray(small, dtype=np.float32) / 255.0
            stat = ImageStat.Stat(small)
            for idx, ch in enumerate(['r', 'g', 'b']):
                row[f'{ch}_mean'] = stat.mean[idx] / 255.0
                row[f'{ch}_std'] = stat.stddev[idx] / 255.0
                row[f'{ch}_p10'] = np.quantile(arr[:, :, idx], 0.10)
                row[f'{ch}_p50'] = np.quantile(arr[:, :, idx], 0.50)
                row[f'{ch}_p90'] = np.quantile(arr[:, :, idx], 0.90)
            red, green, blue = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
            row['brightness'] = arr.mean()
            row['contrast'] = arr.std()
            row['excess_green'] = np.mean(2 * green - red - blue)
            row['green_red_ratio'] = np.mean(green / (red + 1e-4))
            row['visible_ndvi_proxy'] = np.mean((green - red) / (green + red + 1e-4))
        except Exception as exc:
            row['image_error'] = str(exc)
        rows.append(row)

    features = pd.DataFrame(rows)
    features.to_csv(cache_path, index=False)
    return features

train_image_features = extract_image_features(train, 'train_image_features.csv')
test_image_features = extract_image_features(test, 'test_image_features.csv')
image_eda = image_df.merge(train_image_features, on='image_path', how='left')

display(train_image_features.head())
print('Image feature columns:', train_image_features.shape[1])

# 13. Image Feature Signal

In [ ]:
image_feature_cols = [
    'width', 'height', 'brightness', 'contrast', 'excess_green', 'green_red_ratio', 'visible_ndvi_proxy',
    'r_mean', 'g_mean', 'b_mean', 'r_std', 'g_std', 'b_std'
]
image_feature_cols = [c for c in image_feature_cols if c in image_eda.columns]
image_signal = image_eda[image_feature_cols + TARGET_ORDER].corr(method='spearman').loc[image_feature_cols, TARGET_ORDER]
display(image_signal.round(3).sort_values('Dry_Total_g', ascending=False))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(image_signal, annot=True, fmt='.2f', cmap=PALETTE_NAME, center=0, linewidths=0.5, ax=ax)
ax.set_title('13.1 Image Feature Spearman Correlation With Targets')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, feat in zip(axes, ['excess_green', 'visible_ndvi_proxy', 'brightness']):
    if feat in image_eda:
        sns.scatterplot(data=image_eda, x=feat, y='Dry_Total_g', hue='State', palette=PALETTE_NAME, ax=ax)
        ax.set_title(f'13.2 {feat} vs Dry Total')
plt.tight_layout()
plt.show()

# 14. Visual Audit Samples

In [ ]:
def show_image_grid(df, title, sort_col=None, n=12):
    sample = df.copy()
    if sort_col is not None and sort_col in sample:
        sample = sample.sort_values(sort_col, ascending=False).head(n)
    else:
        sample = sample.sample(min(n, len(sample)), random_state=42)

    cols = 4
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.6 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = Image.open(DATA_DIR / row['image_path']).convert('RGB')
        ax.imshow(img)
        label = f"{Path(row['image_path']).name}\nTotal={row.get('Dry_Total_g', np.nan):.1f} | GDM={row.get('GDM_g', np.nan):.1f}"
        ax.set_title(label, fontsize=9)
        ax.axis('off')
    for ax in axes[len(sample):]:
        ax.axis('off')
    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()

show_image_grid(image_eda, '14.1 Highest Dry Total Images', sort_col='Dry_Total_g', n=12)
show_image_grid(image_eda, '14.2 Highest Dead Matter Images', sort_col='Dry_Dead_g', n=12)
show_image_grid(image_eda, '14.3 Random Image Audit', n=12)

# 15. Outlier And Edge Case Review

In [ ]:
outlier_frames = []
for target in TARGET_ORDER:
    q01, q99 = image_df[target].quantile([0.01, 0.99])
    tmp = image_df[(image_df[target] <= q01) | (image_df[target] >= q99)].copy()
    tmp['review_target'] = target
    tmp['tail'] = np.where(tmp[target] >= q99, 'high', 'low')
    tmp['review_value'] = tmp[target]
    outlier_frames.append(tmp[['image_path', 'State', 'Species', 'month', 'review_target', 'tail', 'review_value']])

outliers = pd.concat(outlier_frames, ignore_index=True).sort_values(['review_target', 'tail', 'review_value'])
display(outliers.head(30))
display(outliers.tail(30))
outliers.to_csv(OUTPUT_DIR / 'eda_outlier_review.csv', index=False)
print(f'Wrote {OUTPUT_DIR / "eda_outlier_review.csv"}')

# 16. Lightweight Baseline For Error Diagnostics

This is not the main modeling notebook. It exists here to show which targets and segments are difficult under a simple metadata + image-feature model.

In [ ]:
model_df = image_eda.copy()
model_df['sample_month'] = model_df['Sampling_Date'].dt.month
model_df['sample_dayofyear'] = model_df['Sampling_Date'].dt.dayofyear

base_features = [
    'State', 'Species', 'primary_species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'sample_month', 'sample_dayofyear',
    'brightness', 'contrast', 'excess_green', 'green_red_ratio', 'visible_ndvi_proxy',
    'r_mean', 'g_mean', 'b_mean', 'r_std', 'g_std', 'b_std'
]
base_features = [c for c in base_features if c in model_df.columns]
cat_features = [c for c in ['State', 'Species', 'primary_species'] if c in base_features]
num_features = [c for c in base_features if c not in cat_features]

preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=3)),
    ]), cat_features),
])

def make_model(seed):
    return Pipeline([
        ('preprocess', preprocess),
        ('model', HistGradientBoostingRegressor(max_iter=250, learning_rate=0.06, l2_regularization=0.05, random_state=seed)),
    ])

oof_parts = []
gkf = GroupKFold(n_splits=5)
for target in TARGET_ORDER:
    y = model_df[target]
    preds = np.zeros(len(model_df))
    for fold, (trn_idx, val_idx) in enumerate(gkf.split(model_df, groups=model_df['image_path']), start=1):
        model = make_model(100 + fold)
        model.fit(model_df.iloc[trn_idx][base_features], y.iloc[trn_idx])
        preds[val_idx] = np.clip(model.predict(model_df.iloc[val_idx][base_features]), 0, None)
    oof_parts.append(pd.DataFrame({
        'image_path': model_df['image_path'],
        'State': model_df['State'],
        'primary_species': model_df['primary_species'],
        'month': model_df['month'],
        'target_name': target,
        'target': y,
        'prediction': preds,
        'weight': TARGET_WEIGHTS[target],
    }))

oof = pd.concat(oof_parts, ignore_index=True)
oof['error'] = oof['prediction'] - oof['target']
oof['abs_error'] = oof['error'].abs()
score = weighted_r2_score(oof['target'], oof['prediction'], oof['weight'])
print(f'OOF weighted R2: {score:.5f}')
display(oof.groupby('target_name').agg(mae=('abs_error', 'mean'), bias=('error', 'mean'), target_mean=('target', 'mean')).reindex(TARGET_ORDER).round(3))

# 17. Error Diagnostics By Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=oof, x='target_name', y='abs_error', hue='target_name', palette=TARGET_PALETTE, legend=False, ax=axes[0])
axes[0].set_title('17.1 MAE By Target')
axes[0].tick_params(axis='x', rotation=25)

sns.scatterplot(data=oof, x='target', y='prediction', hue='target_name', palette=TARGET_PALETTE, s=18, ax=axes[1])
limit = max(oof['target'].max(), oof['prediction'].max())
axes[1].plot([0, limit], [0, limit], color='black', linewidth=1)
axes[1].set_title('17.2 OOF Prediction vs Target')
plt.tight_layout()
plt.show()

segment_error = (
    oof.groupby(['target_name', 'State'])
    .agg(rows=('target', 'size'), mae=('abs_error', 'mean'), bias=('error', 'mean'), target_mean=('target', 'mean'))
    .reset_index()
    .sort_values(['target_name', 'mae'], ascending=[True, False])
)
display(segment_error.round(3))
segment_error.to_csv(OUTPUT_DIR / 'eda_segment_error.csv', index=False)
print(f'Wrote {OUTPUT_DIR / "eda_segment_error.csv"}')

# 18. Auto-Generated Insights

Run this section after the EDA cells. It converts the tables above into a compact checklist for modeling decisions and output review.

In [ ]:
insights = []

largest_weight = metric_view['effective_weight_share'].idxmax()
insights.append(f'Metric focus: {largest_weight} has the largest effective score weight share ({metric_view.loc[largest_weight, "effective_weight_share"]:.1%}).')

most_variable = target_summary['std'].idxmax()
insights.append(f'Target variance: {most_variable} has the widest absolute spread, so errors there may dominate raw MAE.')

median_gap = image_df['total_minus_components'].median()
q95_gap = image_df['total_minus_components'].abs().quantile(0.95)
insights.append(f'Accounting check: median Dry_Total minus component sum is {median_gap:.2f}; 95% absolute gap is {q95_gap:.2f}.')

for feature in ['Pre_GSHH_NDVI', 'Height_Ave_cm']:
    if feature in numeric_signal.index:
        best_target = numeric_signal.loc[feature].abs().idxmax()
        corr_value = numeric_signal.loc[feature, best_target]
        insights.append(f'Metadata signal: {feature} is most associated with {best_target} by Spearman correlation ({corr_value:.2f}).')

if 'excess_green' in image_signal.index:
    best_target = image_signal.loc['excess_green'].abs().idxmax()
    insights.append(f'Image signal: excess_green is strongest against {best_target} ({image_signal.loc["excess_green", best_target]:.2f}).')

hardest_target = oof.groupby('target_name')['abs_error'].mean().idxmax()
insights.append(f'Baseline diagnostics: {hardest_target} has the highest OOF MAE under the lightweight model.')

for item in insights:
    print('-', item)

pd.Series(insights, name='insight').to_csv(OUTPUT_DIR / 'eda_insights.csv', index=False)
print(f'Wrote {OUTPUT_DIR / "eda_insights.csv"}')